In [2]:

import duckdb
import pandas as pd

# Use read_only=True so we don't get locked out!
conn = duckdb.connect('../data/churn_database.duckdb', read_only=True)


In [3]:
query1 = """
SELECT 
    CASE 
        WHEN monthlycharges < 50 THEN 'Low Spend (<$50)'
        WHEN monthlycharges BETWEEN 50 AND 100 THEN 'Mid Spend ($50-$100)'
        ELSE 'High Spend (>$100)'
    END as spend_segment,
    COUNT(*) as customer_count,
    ROUND(AVG(churn) * 100, 2) as churn_rate_pct
FROM customers
GROUP BY spend_segment
ORDER BY customer_count DESC;
"""
df_spend = conn.execute(query1).df()
display(df_spend)


,spend_segment,customer_count,churn_rate_pct
0,Mid Spend ($50-$100),684494,10.07
1,High Spend (>$100),259285,9.22
2,Low Spend (<$50),56221,11.33


In [4]:
query2 = """
WITH RiskScoring AS (
    SELECT 
        customer_id,
        churn,
        CASE 
            WHEN contract = 'month_to_month' AND payment_method = 'electronic_check' THEN 'High Risk'
            WHEN contract = 'month_to_month' THEN 'Medium Risk'
            ELSE 'Low Risk'
        END as risk_profile
    FROM customers
)

SELECT 
    risk_profile,
    COUNT(*) as total_customers,
    SUM(churn) as churned_customers,
    ROUND(AVG(churn) * 100, 2) as churn_rate_pct
FROM RiskScoring
GROUP BY risk_profile
ORDER BY churn_rate_pct DESC;
"""
df_risk = conn.execute(query2).df()
display(df_risk)


,risk_profile,total_customers,churned_customers,churn_rate_pct
0,Medium Risk,15031,4000.0,26.61
1,High Risk,4961,1299.0,26.18
2,Low Risk,980008,93928.0,9.58
